In [1]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms

import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

os.chdir('/content/drive/MyDrive/Project/BrainRegionId/Project43/Code')
from modules.networks_reg import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *

In [2]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-794d179e-be30-3dcc-c07c-3c8e96f5b9b9)


In [3]:
def net_train_AnyNet_L_reg(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.MSELoss(reduction='mean')
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print(f'lr: {lr}')
    loss_train_array = []
    loss_valid_array = []
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        loss_train = []
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            coordinate0_train = Classifier(x_train.to(device))
            L = loss_fn(coordinate0_train, coordinate_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print(f'MSE loss: {L.item()}')
            loss_train.append(L.item())
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            epoch0 += 1
        print(f'Loss Train: {torch.mean(torch.tensor(loss_train))}')

        loss_train_array.append(torch.mean(torch.tensor(loss_train)))

        Classifier.eval()
        loss_valid = []
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            coordinate0_valid = Classifier(x_valid.to(device))
            L_valid = loss_fn(coordinate0_valid, coordinate_valid.to(device))
            del x_valid, x_valid1
            loss_valid.append(L_valid.item())

        print(f'Loss valid: {torch.mean(torch.tensor(loss_valid))}')

        if epoch == 0:
            acu_valid_m = torch.mean(torch.tensor(loss_valid)).detach()
        else:
            if torch.mean(torch.tensor(loss_valid)) < acu_valid_m:
                torch.save(Classifier, train_args['save_dir'] + f'/AnyNet_L_{key}_reg{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/AnyNet_L_{key}_reg_epoch{ind}.pt')
                # return

        loss_valid_array.append(torch.mean(torch.tensor(loss_valid)))

    torch.save(loss_train_array, train_args['save_dir'] + f'/AnyNet_L_{key}_train_acu{ind}.pt')
    torch.save(loss_valid_array, train_args['save_dir'] + f'/AnyNet_L_{key}_valid_acu{ind}.pt')

    return

def net_train_ViT_L_reg(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.MSELoss(reduction='mean')
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print(f'lr: {lr}')
    loss_train_array = []
    loss_valid_array = []
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        loss_train = []
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            coordinate0_train = Classifier(x_train.to(device))
            L = loss_fn(coordinate0_train, coordinate_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print(f'MSE loss: {L.item()}')
            loss_train.append(L.item())
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            epoch0 += 1
        print(f'Loss Train: {torch.mean(torch.tensor(loss_train))}')

        loss_train_array.append(torch.mean(torch.tensor(loss_train)))

        Classifier.eval()
        loss_valid = []
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            coordinate0_valid = Classifier(x_valid.to(device))
            L_valid = loss_fn(coordinate0_valid, coordinate_valid.to(device))
            del x_valid, x_valid1
            loss_valid.append(L_valid.item())

        print(f'Loss valid: {torch.mean(torch.tensor(loss_valid))}')

        if epoch == 0:
            acu_valid_m = torch.mean(torch.tensor(loss_valid)).detach()
        else:
            if torch.mean(torch.tensor(loss_valid)) < acu_valid_m:
                torch.save(Classifier, train_args['save_dir'] + f'/ViT_L_{key}_reg{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/ViT_L_{key}_reg_epoch{ind}.pt')
                # return

        loss_valid_array.append(torch.mean(torch.tensor(loss_valid)))

    torch.save(loss_train_array, train_args['save_dir'] + f'/ViT_L_{key}_train_acu{ind}.pt')
    torch.save(loss_valid_array, train_args['save_dir'] + f'/ViT_L_{key}_valid_acu{ind}.pt')

    return

def net_train_RNN_L_reg(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.MSELoss(reduction='mean')
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print(f'lr: {lr}')
    loss_train_array = []
    loss_valid_array = []
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        loss_train = []
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args).squeeze(1).permute(0, 2, 1)
            y_train = y_train.to(device)
            coordinate0_train = Classifier(x_train.to(device))
            L = loss_fn(coordinate0_train, coordinate_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print(f'MSE loss: {L.item()}')
            loss_train.append(L.item())
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            epoch0 += 1
        print(f'Loss Train: {torch.mean(torch.tensor(loss_train))}')

        loss_train_array.append(torch.mean(torch.tensor(loss_train)))

        Classifier.eval()
        loss_valid = []
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args).squeeze(1).permute(0, 2, 1)
            y_valid = y_valid.to(device)
            coordinate0_valid = Classifier(x_valid.to(device))
            L_valid = loss_fn(coordinate0_valid, coordinate_valid.to(device))
            del x_valid, x_valid1
            loss_valid.append(L_valid.item())

        print(f'Loss valid: {torch.mean(torch.tensor(loss_valid))}')

        if epoch == 0:
            acu_valid_m = torch.mean(torch.tensor(loss_valid)).detach()
        else:
            if torch.mean(torch.tensor(loss_valid)) < acu_valid_m:
                torch.save(Classifier, train_args['save_dir'] + f'/RNN_L_{key}_reg{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/RNN_L_{key}_reg_epoch{ind}.pt')
                # return

        loss_valid_array.append(torch.mean(torch.tensor(loss_valid)))

    torch.save(loss_train_array, train_args['save_dir'] + f'/RNN_L_{key}_train_acu{ind}.pt')
    torch.save(loss_valid_array, train_args['save_dir'] + f'/RNN_L_{key}_valid_acu{ind}.pt')

    return

In [4]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [5]:
# @title Load data
file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
brain_signal_lfp = torch.load(file_dir + '/brain_signal_lfp1.pt')
for file_id in [2, 3, 4, 5]:
    brain_signal_lfp = torch.concat([brain_signal_lfp, torch.load(file_dir + f'/brain_signal_lfp{file_id}.pt')], dim=0)
    print(f'Load file id: {file_id}')

# torch.save(brain_signal_lfp, '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/brain_signal_lfp')
list_dict = torch.load(file_dir + '/list_dict.pt', weights_only=False)
# list_dict_acronym_selec = torch.load(file_dir + '/list_dict_acronym_selec.pt')
brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']

Load file id: 2
Load file id: 3
Load file id: 4
Load file id: 5


In [6]:
if len(brain_signal_lfp) == len(brain_region_index):
    print('Matched, no damage!')

Matched, no damage!


In [7]:
list_dict.keys()

dict_keys(['brain_region_index', 'brain_region_index_Cosmos', 'brain_region_atlas', 'subject_list', 'exp_list', 'key_list', 'coordinate_list', 'depth_list', 'volume_list', 'brain_signal_lfp', 'brain_signal_ap', 'train_list_key', 'test_list_key', 'train_list_subject', 'test_list_subject', 'train_list_exp', 'test_list_exp', 'train_list_trial', 'test_list_trial', 'train_list_intest', 'test_list_intest', 'acronym_selec_list', 'valid_list_intest'])

In [8]:
subject_num = 10
key = 'stimOff_times'
subject_od_ind = subject_od_ind_gen(list_dict, acronym_list, subject_num)
train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)

data_train = TensorDataset(brain_signal_lfp[train_ind,:], brain_region_index[train_ind], coordinate_list[train_ind])
data_valid = TensorDataset(brain_signal_lfp[valid_ind,:], brain_region_index[valid_ind], coordinate_list[valid_ind])
data_test = TensorDataset(brain_signal_lfp[test_ind,:], brain_region_index[test_ind], coordinate_list[test_ind])

train_iter = DataLoader(data_train, batch_size=128, shuffle=True)
valid_iter = DataLoader(data_valid, batch_size=128, shuffle=True)
test_iter = DataLoader(data_test, batch_size=128, shuffle=True)

FRP1
FRP2/3
FRP5
FRP6a
MOp1
MOp2/3
MOp5
MOp6a
MOp6b
MOs1
MOs2/3
MOs5
MOs6a
MOs6b
SSp-n1
SSp-n2/3
SSp-n4
SSp-n5
SSp-n6a
SSp-n6b
SSp-bfd1
SSp-bfd2/3
SSp-bfd4
SSp-bfd5
SSp-bfd6a
SSp-bfd6b
SSp-ll1
SSp-ll2/3
SSp-ll4
SSp-ll5
SSp-ll6a
SSp-ll6b
SSp-m1
SSp-m2/3
SSp-m4
SSp-m5
SSp-m6a
SSp-m6b
SSp-ul1
SSp-ul2/3
SSp-ul4
SSp-ul5
SSp-ul6a
SSp-ul6b
SSp-tr1
SSp-tr2/3
SSp-tr4
SSp-tr5
SSp-tr6a
SSp-tr6b
SSp-un1
SSp-un2/3
SSp-un4
SSp-un5
SSp-un6a
SSp-un6b
SSs2/3
SSs4
SSs5
SSs6a
SSs6b
GU5
GU6a
VISC5
VISC6a
VISC6b
AUDd2/3
AUDd4
AUDd5
AUDd6a
AUDd6b
AUDp4
AUDp5
AUDp6a
AUDpo2/3
AUDpo4
AUDpo5
AUDpo6a
AUDpo6b
AUDv5
AUDv6a
AUDv6b
VISal2/3
VISal4
VISal5
VISal6a
VISal6b
VISam1
VISam2/3
VISam4
VISam5
VISam6a
VISam6b
VISl1
VISl2/3
VISl4
VISl5
VISl6a
VISl6b
VISp1
VISp2/3
VISp4
VISp5
VISp6a
VISp6b
VISpl1
VISpl2/3
VISpl4
VISpl5
VISpl6a
VISpm1
VISpm2/3
VISpm4
VISpm5
VISpm6a
VISpm6b
VISli2/3
VISli4
VISli5
VISli6a
VISli6b
VISpor1
VISpor2/3
VISpor5
VISpor6a
VISpor6b
ACAd2/3
ACAd5
ACAd6a
ACAd6b
ACAv1
ACAv2/3
ACAv5
ACAv6a
ACAv

In [9]:
save_dir = '/content/drive/MyDrive/Project/BrainRegionId/Science/Model/Reg'

In [ ]:
c0 = 64 * 4
k0 = 1.0

model_args = {
    'arch':((2,c0 * 2,1,k0), (2,c0 * 3,1,k0), (2,c0 * 4,1,k0), (2,c0 * 5,1,k0)),
    'stem_channels':c0,
}
train_args = {
    'overfitting_thres':0.60,
    'lr':5e-4,
    'norm':True,
    'temp':[True, True],
    'epochs':50,
    'save_dir':save_dir,
}

for ind in range(0, 1):
    Classifier = AnyNet_L_reg(model_args['arch'], model_args['stem_channels'], frequency_bin=spectro_args['spectro_img'][0], num_classes=len(acronym_list)).to(device)
    Classifier.apply(init_cnn)
    net_train_AnyNet_L_reg(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

MSE loss: 0.274053692817688
MSE loss: 0.0010602229740470648
MSE loss: 7.652828207938e-05
MSE loss: 0.00023900278029032052
MSE loss: 0.0009209974086843431
MSE loss: 0.0018450268544256687
MSE loss: 0.00048382487148046494
MSE loss: 0.0027022892609238625
MSE loss: 0.0005070724291726947
MSE loss: 0.0001969213772099465
MSE loss: 0.0008074573124758899
MSE loss: 0.0006666097324341536
MSE loss: 0.00019982107914984226
Loss Train: 0.0015566013753414154
Loss valid: 0.0016052965074777603
MSE loss: 0.0058535244315862656
MSE loss: 0.00011719826579792425
MSE loss: 8.810199506115168e-05
MSE loss: 0.0024308105930685997
MSE loss: 0.0001545865961816162
MSE loss: 4.585067472362425e-06
MSE loss: 4.965248444932513e-05
MSE loss: 0.00014604852185584605
MSE loss: 9.253765165340155e-05
MSE loss: 4.5817341742804274e-05
MSE loss: 0.00018918805290013552
MSE loss: 0.00021097171702422202
MSE loss: 0.0007362121250480413
Loss Train: 0.00035220777499489486
Loss valid: 0.00019045115914195776
MSE loss: 0.00023187992337625

In [ ]:
for ind in range(0, 1):
    img_size, patch_size = (224, 28), (28, 28)
    num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 4
    emb_dropout, blk_dropout = 0.1, 0.1
    Classifier = ViT_L_reg(spectro_args['spectro_img'][0], img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout, num_classes=len(acronym_list)).to(device)
    net_train_ViT_L_reg(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

MSE loss: 0.2209949940443039
MSE loss: 0.0006492533721029758
MSE loss: 0.00025862123584374785
MSE loss: 0.0003262989630457014
MSE loss: 0.00012152339331805706
MSE loss: 0.0001766391796991229
MSE loss: 0.00030833864002488554
MSE loss: 6.735640636179596e-05
MSE loss: 5.9109352150699124e-05
MSE loss: 0.00030706857796758413
MSE loss: 0.00013649975880980492
MSE loss: 1.8775826902128756e-05
MSE loss: 0.0004021278000436723
Loss Train: 0.00482720322906971
Loss valid: 6.953124102437869e-05
MSE loss: 7.650269253645092e-05
MSE loss: 3.0207225790945813e-05
MSE loss: 5.650863749906421e-05
MSE loss: 4.436082599568181e-05
MSE loss: 5.726336530642584e-05
MSE loss: 8.089489710982889e-05
MSE loss: 7.232428470160812e-05
MSE loss: 2.769769707811065e-05
MSE loss: 2.900059007515665e-05
MSE loss: 6.093295451137237e-05
MSE loss: 4.99587767990306e-05
MSE loss: 5.574897659244016e-05
MSE loss: 7.899972843006253e-05
Loss Train: 5.1405491831246763e-05
Loss valid: 1.1498620551719796e-05
MSE loss: 1.1016672033292707

In [ ]:
RNN_args = {
    'input_size':224,
    'hidden_size':512 * 2,
    'num_layers':2,
    'category_num':len(acronym_list),
    'data_len':28,
}
for ind in range(0, 1):
    Classifier = RNN_L_reg(spectro_args['spectro_img'][0], RNN_args).to(device)
    net_train_RNN_L_reg(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

MSE loss: 0.000868653878569603
MSE loss: 3.2084822123579215e-06
MSE loss: 3.878246843669331e-06
MSE loss: 3.544390438037226e-06
MSE loss: 3.707555606524693e-06
MSE loss: 4.2536298678896856e-06
MSE loss: 3.4680404041864676e-06
MSE loss: 3.867618033837061e-06
MSE loss: 3.838596057903487e-06
MSE loss: 2.951176384158316e-06
MSE loss: 3.059094069612911e-06
MSE loss: 3.663081542981672e-06
MSE loss: 3.798997113335645e-06
Loss Train: 8.434020855929703e-06
Loss valid: 5.104581759951543e-06
MSE loss: 3.1564802611683263e-06
MSE loss: 2.989082304338808e-06
MSE loss: 3.1982435757527128e-06
MSE loss: 2.8233225748408586e-06
MSE loss: 3.0659775802632794e-06
MSE loss: 2.7901833163923584e-06
MSE loss: 3.4871188745455584e-06
MSE loss: 2.7960693387285573e-06
MSE loss: 3.234793211959186e-06
MSE loss: 2.6415307274874067e-06
MSE loss: 2.911066076194402e-06
MSE loss: 2.8150548132543918e-06
MSE loss: 2.392897386016557e-06
Loss Train: 3.037727992705186e-06
Loss valid: 2.898710818044492e-06
MSE loss: 3.095588908

In [ ]:
from google.colab import runtime
runtime.unassign()